# 🎬 NovaCraft AI: Text-to-Video Custom LoRA Training
### Complete Pipeline: Telegram Dataset ➔ Qwen2-VL Auto-Captioning ➔ Video LoRA Training ➔ Telegram Delivery

**Hardware Support:** Google Colab Free T4 (16GB) ya Colab Pro A100 (40GB)
**Base Model:** Wan 2.1 / CogVideoX via Hugging Face Diffusers

## 1️⃣ Environment & Dependencies Install Karein

In [ ]:
!nvidia-smi

# Install core training and vision AI packages
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate peft bitsandbytes decord opencv-python imageio imageio-ffmpeg
!pip install -q qwen-vl-utils torchvision
!pip install -q python-dotenv requests

print("✅ Sabhi Dependencies successfully install ho gayi hain!")

## 2️⃣ Telegram Dataset Downloader
Apne Telegram channel se sabhi uploaded videos aur unke captions Colab me fetch karein.

In [ ]:
import os
import requests
import json

# Apne Telegram credentials enter karein
BOT_TOKEN = "8469994558:AAH_Hr4pxr1nTnMuRKXzBIOgUVrYqo3dQ10"
CHANNEL_ID = "-1004461650802"

DATASET_DIR = "/content/dataset/videos"
os.makedirs(DATASET_DIR, exist_ok=True)

print(f"📡 Fetching videos from Telegram Channel: {CHANNEL_ID}...")

# Fetch updates from Telegram
res = requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/getUpdates?limit=100").json()
updates = res.get("result", [])

raw_dataset = []
video_idx = 1

for u in updates:
    msg = u.get("message") or u.get("channel_post")
    if not msg: continue
    
    video_obj = msg.get("video") or msg.get("animation")
    if video_obj:
        file_id = video_obj["file_id"]
        caption = msg.get("caption", "").strip()
        
        # Get direct download URL
        file_info = requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/getFile?file_id={file_id}").json()
        if file_info.get("ok"):
            file_path = file_info["result"]["file_path"]
            download_url = f"https://api.telegram.org/file/bot{BOT_TOKEN}/{file_path}"
            
            save_name = f"clip_{video_idx:04d}.mp4"
            target_path = os.path.join(DATASET_DIR, save_name)
            
            r = requests.get(download_url, stream=True)
            with open(target_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            raw_dataset.append({
                "video_path": target_path,
                "caption": caption
            })
            print(f"📥 Downloaded {save_name} (Caption: '{caption[:40]}...')")
            video_idx += 1

print(f"\n🎉 Total {len(raw_dataset)} video(s) download ho gayi hain!")

## 3️⃣ Video Standardization & Qwen2-VL Auto-Captioning
Agar kisi video par caption nahi likha tha, toh **Qwen2-VL Vision AI** video ke frames dekh kar ultra-rich prompt generate karega.

In [ ]:
import cv2
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Check if any video needs auto-captioning
needs_captioning = [item for item in raw_dataset if not item["caption"] or len(item["caption"]) < 5]

if needs_captioning:
    print(f"🧠 Loading Qwen2-VL-2B-Instruct to describe {len(needs_captioning)} video(s)...")
    model_id = "Qwen/Qwen2-VL-2B-Instruct"
    processor = AutoProcessor.from_pretrained(model_id)
    vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    for item in raw_dataset:
        if not item["caption"] or len(item["caption"]) < 5:
            # Extract 4 keyframes from video
            cap = cv2.VideoCapture(item["video_path"])
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            step = max(total_frames // 4, 1)
            frames = []
            for i in range(4):
                cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
                ret, frame = cap.read()
                if ret:
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(Image.fromarray(frame_rgb))
            cap.release()

            if frames:
                prompt = "Describe this video in detail for AI text-to-video training: mention subject, action, lighting, camera movement, style in 1 concise sentence."
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": frames[0]},
                            {"type": "image", "image": frames[2]},
                            {"type": "text", "text": prompt}
                        ]
                    }
                ]
                text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                image_inputs, _ = process_vision_info(messages)
                device = "cuda" if torch.cuda.is_available() else "cpu"
                inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(device)
                generated_ids = vlm_model.generate(**inputs, max_new_tokens=60)
                caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].split("assistant\n")[-1].strip()
                item["caption"] = caption
                print(f"✨ [Auto-Prompt Generated for {os.path.basename(item['video_path'])}]: {caption}")

    # Free VLM memory for diffusion training
    del vlm_model
    del processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Vision AI memory cleared for Diffusion training!")

# Save final dataset manifest
manifest_file = "/content/dataset/metadata.json"
with open(manifest_file, "w") as f:
    json.dump(raw_dataset, f, indent=2)

print(f"✅ Dataset Manifest Ready: {manifest_file} with {len(raw_dataset)} training pairs!")

## 4️⃣ Video LoRA Training Engine (Diffusers + PEFT)
Temporal & Spatial Layers ko fine-tune karke LoRA weights banayein.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model

OUTPUT_DIR = "/content/novacraft_video_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LoRA Configuration for Video Diffusion
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.05,
    bias="none",
)

print("🚀 LoRA Adapter Config Ready!")
print(f"📊 Training Rank (r): {lora_config.r}, Alpha: {lora_config.lora_alpha}")
print(f"💾 Output Directory: {OUTPUT_DIR}")

## 5️⃣ Generate Test Video & Send to Telegram Channel
Trained LoRA model se nayi video create karein aur Telegram me preview bhejein.

In [ ]:
def send_video_to_telegram(video_path, caption_text):
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendVideo"
    with open(video_path, "rb") as f:
        files = {"video": f}
        data = {
            "chat_id": CHANNEL_ID,
            "caption": caption_text,
            "parse_mode": "HTML"
        }
        r = requests.post(url, files=files, data=data)
        if r.status_code == 200:
            print("🎉 Video successfully sent to Telegram Channel!")
        else:
            print(f"Telegram send error: {r.text}")

print("✅ Telegram Delivery Hook Active!")